In [ ]:
# synthetic_data_generator.py
import pandas as pd
import numpy as np
import random
import json # Import json for jsonl saving

def generate_readmission_data(num_records=500):
    """Generates synthetic data for patient readmission prediction."""
    # Define potential patient characteristics
    genders = ['Male', 'Female', 'Other']
    discharge_dispositions = ['Home', 'Skilled Nursing Facility', 'Expired', 'Other Facility']
    
    # Generate base data
    data = {
        'PatientID': [f'PID_{i:04d}' for i in range(num_records)],
        'Age': np.random.randint(18, 95, num_records), # Age range 18-95
        'Gender': np.random.choice(genders, num_records, p=[0.47, 0.51, 0.02]), # Probabilities for gender distribution
        'NumPriorAdmissions': np.random.poisson(1.8, num_records), # Average prior admissions around 1.8
        'HasDiabetes': np.random.choice([0, 1], num_records, p=[0.65, 0.35]), # 35% have diabetes
        'HasHypertension': np.random.choice([0, 1], num_records, p=[0.55, 0.45]), # 45% have hypertension
        'HasHeartDisease': np.random.choice([0, 1], num_records, p=[0.80, 0.20]), # 20% have heart disease
        'LengthOfStay': np.random.gamma(shape=3, scale=2.5, size=num_records).astype(int) + 1, # Skewed length of stay (min 1 day)
        'DischargeDisposition': np.random.choice(discharge_dispositions, num_records, p=[0.70, 0.15, 0.05, 0.10]), # Discharge distribution
    }
    df = pd.DataFrame(data)

    # Simple probabilistic logic for readmission based on factors
    # Base probability increases with age, prior admissions, and comorbidities
    readmission_prob = (
        0.05 + # Base chance
        (df['Age'] / 300) + # Increase with age
        (df['NumPriorAdmissions'] * 0.08) + # Increase with prior admissions
        (df['HasDiabetes'] * 0.06) +
        (df['HasHypertension'] * 0.04) +
        (df['HasHeartDisease'] * 0.07) +
        (df['LengthOfStay'] / 100) # Slight increase with longer stay
    )
    # Cap probability at 0.9
    readmission_prob = np.clip(readmission_prob, 0, 0.9)
    
    # Determine readmission based on calculated probability
    df['ReadmittedWithin30Days'] = (np.random.rand(num_records) < readmission_prob).astype(int)

    # Ensure 'Expired' patients are not readmitted
    df.loc[df['DischargeDisposition'] == 'Expired', 'ReadmittedWithin30Days'] = 0

    return df

def generate_healthcare_qa_data(num_records=50):
    """Generates synthetic data for healthcare Q&A evaluation (JSONL format)."""
    # Expanded contexts and Q&A pairs
    qa_pairs = [
        {"context": "Aspirin is commonly used to reduce pain, fever, and inflammation. Low doses may reduce heart attack/stroke risk. Side effects include stomach upset. Avoid giving to children with viral illnesses (Reye's syndrome risk).",
         "questions": [
             {"q": "What are the main uses of aspirin?", "a": "Aspirin is used for pain, fever, inflammation, and sometimes heart attack/stroke prevention."},
             {"q": "Is aspirin safe for kids?", "a": "No, children with viral illnesses should not take aspirin due to the risk of Reye's syndrome."},
             {"q": "What is a common side effect of aspirin?", "a": "A common side effect is stomach upset."}
         ]},
        {"context": "Metformin is a primary drug for type 2 diabetes. It lowers liver glucose production and boosts insulin sensitivity. Common side effects: diarrhea, nausea. Take with meals.",
         "questions": [
             {"q": "How does Metformin help manage diabetes?", "a": "It works by reducing glucose production in the liver and increasing the body's sensitivity to insulin."},
             {"q": "What type of diabetes is Metformin used for?", "a": "Metformin is primarily used for type 2 diabetes."},
             {"q": "What are typical side effects of Metformin?", "a": "Diarrhea and nausea are common side effects."}
         ]},
        {"context": "Hypertension (high blood pressure) often lacks symptoms. Regular checks are vital. Management involves lifestyle (low sodium diet, exercise, weight loss) and potentially medication.",
         "questions": [
             {"q": "Does high blood pressure usually have symptoms?", "a": "No, hypertension often has no symptoms, making regular checks important."},
             {"q": "What lifestyle changes help manage hypertension?", "a": "Low sodium diet, regular exercise, and weight management are key lifestyle changes."},
             {"q": "Besides lifestyle, how is high blood pressure treated?", "a": "Medications may be prescribed if lifestyle changes are insufficient."}
         ]},
         {"context": "Atorvastatin (Lipitor) is a statin medication used to lower cholesterol and triglycerides in the blood. It helps prevent heart attacks and strokes by reducing 'bad' cholesterol (LDL). Muscle pain can be a side effect.",
         "questions": [
             {"q": "What is Atorvastatin used for?", "a": "It is used to lower cholesterol and triglycerides, helping prevent heart attacks and strokes."},
             {"q": "What type of cholesterol does Atorvastatin primarily reduce?", "a": "It primarily reduces LDL or 'bad' cholesterol."},
             {"q": "Is muscle pain a known side effect of Atorvastatin?", "a": "Yes, muscle pain can be a side effect of statins like Atorvastatin."}
         ]}
    ]

    output_records = []
    for i in range(num_records):
        # Select a random context block
        chosen_block = random.choice(qa_pairs)
        context = chosen_block["context"]
        # Select a random question/answer pair from that block
        chosen_qa = random.choice(chosen_block["questions"])
        question = chosen_qa["q"]
        ground_truth = chosen_qa["a"]

        output_records.append({
            "context": context,
            "question": question,
            "ground_truth": ground_truth # Ground truth answer based on the context
        })

    # Save as JSON Lines (.jsonl)
    output_filename = 'healthcare_qa_eval_data.jsonl'
    try:
        with open(output_filename, 'w') as f:
            for record in output_records:
                # Use json.dumps for proper JSON formatting of each line
                f.write(json.dumps(record) + '\n')
        print(f"Generated {output_filename}")
    except IOError as e:
        print(f"Error writing file {output_filename}: {e}")
        
    return output_records # Return list of dicts as well

# --- Generate and Save the Data ---
try:
    readmission_df = generate_readmission_data(500)
    readmission_filename = 'synthetic_readmission_data.csv'
    readmission_df.to_csv(readmission_filename, index=False)
    print(f"Generated {readmission_filename}")
except Exception as e:
    print(f"Error generating or saving readmission data: {e}")

try:
    generate_healthcare_qa_data(50)
except Exception as e:
    print(f"Error generating or saving Q&A data: {e}")



In [ ]:
# evaluate_generative_sdk.py
import os
import json
import time
import pandas as pd
from openai import AzureOpenAI, RateLimitError, APIError # Import specific errors

# --- Azure OpenAI Configuration ---
# Ensure these environment variables are set before running the script
AZURE_OPENAI_ENDPOINT = ""
AZURE_OPENAI_KEY = ""
AZURE_OPENAI_DEPLOYMENT_NAME = ""
AZURE_OPENAI_API_VERSION = ""

# --- Evaluation Configuration ---
EVAL_DATA_PATH = "healthcare_qa_eval_data.jsonl" # Input data file
EVALUATION_NAME = "healthcare_qa_bot_eval_sdk" # Name for the evaluation run in Azure AI Studio
OUTPUT_PATH = "./qa_eval_results_sdk" # Local directory to save detailed results

# --- 1. Initialize Azure OpenAI Client ---
client = None
if all([AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_KEY, AZURE_OPENAI_DEPLOYMENT_NAME]):
    try:
        client = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_key=AZURE_OPENAI_KEY,
            api_version=AZURE_OPENAI_API_VERSION
        )
        print("Azure OpenAI client initialized successfully.")
    except Exception as e:
        print(f"Error initializing Azure OpenAI client: {e}. Evaluation cannot proceed without it.")
        exit() # Exit if client setup fails
else:
    print("Error: Azure OpenAI environment variables (ENDPOINT, KEY, DEPLOYMENT_NAME) are not set.")
    print("Evaluation cannot proceed without a connection to the model.")
    exit() # Exit if config is missing

# --- 2. Define Function to Call the Chat Model ---
# This function takes the prompt and returns the model's response string.
# Includes basic error handling and retry logic.
def call_azure_openai_chat(prompt: str, max_retries=3, delay=5) -> str:
    """Calls the configured Azure OpenAI chat deployment with retry logic."""
    if not client:
        return "Error: Azure OpenAI client not initialized."

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Answer the user's question based *only* on the provided context. If the answer is not in the context, say 'The context does not provide an answer to this question.'"},
        {"role": "user", "content": prompt}
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=AZURE_OPENAI_DEPLOYMENT_NAME,
                messages=messages,
                temperature=0.1, # Low temperature for more deterministic, fact-based answers
                max_tokens=200, # Limit response length
                n=1, # Generate one response
                stop=None # No specific stop sequences
            )
            # Ensure response structure is as expected
            if response.choices and len(response.choices) > 0:
                message = response.choices[0].message
                if message and message.content:
                    return message.content.strip()
                else:
                    print(f"Warning: Empty message content received (Attempt {attempt+1})")
                    return "Error: Received empty content from API."
            else:
                 print(f"Warning: No choices received in API response (Attempt {attempt+1})")
                 return "Error: No choices received from API."

        except RateLimitError as e:
            print(f"Rate limit error (Attempt {attempt+1}/{max_retries}): {e}. Retrying in {delay} seconds...")
            if attempt < max_retries - 1:
                time.sleep(delay)
            else:
                print("Max retries reached for rate limit error.")
                return f"Error: Rate limit exceeded after {max_retries} attempts."
        except APIError as e:
            print(f"API error (Attempt {attempt+1}/{max_retries}): {e}. Retrying in {delay} seconds...")
            if attempt < max_retries - 1:
                time.sleep(delay)
            else:
                print("Max retries reached for API error.")
                return f"Error: API error after {max_retries} attempts."
        except Exception as e:
            print(f"An unexpected error occurred (Attempt {attempt+1}/{max_retries}): {e}")
            # Decide if retry makes sense for other errors, here we stop
            return f"Error: Unexpected error calling Azure OpenAI: {e}"
            
    return f"Error: Failed to get response after {max_retries} attempts." # Should only be reached if loop logic fails

# --- 3. Define the Target Function for Evaluation SDK ---
# This function maps a row from the input data to the format expected by the model call,
# calls the model, and returns the output in the format expected by the evaluators.
def model_prediction_target_fn(data_row: dict):
    """
    Target function for the evaluation SDK.
    Input: Dictionary representing one row from the JSONL file (e.g., {'context': '...', 'question': '...'})
    Output: Dictionary containing the model's prediction (e.g., {'answer': '...'})
    """
    context = data_row.get("context", "")
    question = data_row.get("question", "")

    # Construct the prompt for the model
    # It's crucial to clearly separate context and question for the model
    prompt = f"Context:\n---\n{context}\n---\n\nQuestion:\n{question}\n\nAnswer based only on the context provided above:"

    # Call the actual model endpoint using the helper function
    generated_answer = call_azure_openai_chat(prompt)

    # Return the result with the key 'answer', which is the default key expected by many built-in evaluators
    return {"answer": generated_answer}

# --- 4. Set up Azure ML Client Connection (for logging results) ---
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = None # Initialize to None
try:
    credential = DefaultAzureCredential()
    # Check if config.json exists, otherwise specify details
    if os.path.exists("config.json"):
         ml_client = MLClient.from_config(credential=credential)
    else:
         # Replace with your subscription, resource group, and workspace details
         subscription_id = "8855ce15-2cf9-4df6-8f25-33ee12bb4bf3"
         resource_group = ai-demos
         workspace = ai-demos-ml
         ml_client = MLClient(credential, subscription_id, resource_group, workspace)

    print(f"Connected to Azure ML Workspace: {ml_client.workspace_name} (for logging evaluation results)")
except Exception as e:  
    print(f"Warning: Could not connect to Azure ML Workspace: {e}. Evaluation results will only be saved locally.")
    # Continue without ml_client, results won't be logged to the Azure ML service

# --- 5. Run Evaluation using azure-ai-generative SDK ---
# Import necessary components from the generative SDK
from azure.ai.generative.evaluate import evaluate
# Import built-in metrics relevant to Q&A tasks
from azure.ai.generative.evaluate.metrics import (
    Groundedness, # Does the answer rely on the provided context?
    Relevance,    # Is the answer relevant to the question?
    Fluency,      # Is the answer grammatically correct and easy to read?
    Coherence     # Does the answer make logical sense?
)

print(f"\nStarting Generative AI Evaluation: {EVALUATION_NAME}")
print(f"Using data: {EVAL_DATA_PATH}")
print(f"Metrics: Groundedness, Relevance, Fluency, Coherence")

# Define how columns in your data file map to the standard inputs expected by the metrics.
# Keys are the standard metric input names, values are the column names in your JSONL data.
# 'answer' refers to the key in the dictionary returned by `model_prediction_target_fn`.
data_mapping = {
    "question": "question",         # Maps 'question' column in JSONL to metric input 'question'
    "context": "context",           # Maps 'context' column in JSONL to metric input 'context'
    "ground_truth": "ground_truth", # Maps 'ground_truth' column in JSONL to metric input 'ground_truth' (used by some metrics like Exact Match, not used by default GPT metrics here but good practice to include)
    "answer": "answer"              # Maps the 'answer' key from model_prediction_target_fn output to metric input 'answer'
}

# Define the list of metrics to compute
# These metrics use a GPT model (configured in your Azure AI Studio project) to perform the evaluation.
metrics_to_compute = [
    Groundedness(), 
    Relevance(), 
    Fluency(), 
    Coherence()
]

# Execute the evaluation
# The evaluate() function handles iterating through data, calling the target function, 
# calling the evaluation metrics (which may involve calls to a GPT model), and aggregating results.
try:
    evaluation_result = evaluate(
        evaluation_name=EVALUATION_NAME, # Name for the run in Azure AI Studio
        target=model_prediction_target_fn, # Your function that calls the model
        data=EVAL_DATA_PATH,               # Path to the evaluation dataset (JSONL)
        task_type="qa",                    # Specifies the task type (helps select appropriate metrics/defaults)
        metrics_list=metrics_to_compute,   # List of metric objects to compute
        data_mapping=data_mapping,         # How your data columns map to metric inputs
        output_path=OUTPUT_PATH,           # Local directory to save detailed JSONL results
        ml_client=ml_client,               # Pass the MLClient object to log results to Azure ML
        # Optional: Specify the Azure OpenAI connection for the *evaluators* if different from default project connection
        # evaluator_config={ 
        #     "azure_ai_project": {
        #         "subscription_id": ml_client.subscription_id if ml_client else None,
        #         "resource_group_name": ml_client.resource_group_name if ml_client else None,
        #         "project_name": ml_client.workspace_name if ml_client else None, # Assuming project name = workspace name
        #         "connection_name": "Default_AzureOpenAI" # Or your specific connection name
        #     }
        # }
    )
    print("Evaluation finished successfully.")

except Exception as e:
    print(f"\nError during evaluation: {e}")
    evaluation_result = None # Ensure result is None if evaluation fails

# --- 6. Display Summary Results ---
if evaluation_result and evaluation_result.get("metrics_summary"):
    print("\n--- Evaluation Summary ---")
    # Pretty print the summary dictionary
    print(json.dumps(evaluation_result["metrics_summary"], indent=2))

    # Provide guidance on where to find results
    if ml_client and evaluation_result.get("run_info"):
         run_id = evaluation_result['run_info'].get('name', 'N/A')
         print(f"\nView detailed results in Azure AI Studio -> Evaluations -> Run Name: {run_id}")
         # Construct URL (best effort, might need adjustment based on exact Studio URL structure)
         studio_url = f"https://ai.azure.com/projectdeployments/{ml_client.workspace_name}/evaluate/runs/{run_id}?wsid=/subscriptions/{ml_client.subscription_id}/resourcegroups/{ml_client.resource_group_name}/providers/Microsoft.MachineLearningServices/workspaces/{ml_client.workspace_name}"
         print(f"Direct Link (approximate): {studio_url}")


    print(f"\nDetailed results (JSONL) saved locally in: {OUTPUT_PATH}")
    # Example of how to load the detailed results locally
    try:
        detailed_results_file = os.path.join(OUTPUT_PATH, "eval_results.jsonl")
        if os.path.exists(detailed_results_file):
            results_df = pd.read_json(detailed_results_file, lines=True)
            print(f"Loaded detailed results locally, shape: {results_df.shape}")
            # print(results_df.head()) # Uncomment to view first few rows
        else:
             print(f"Detailed results file not found at: {detailed_results_file}")
    except Exception as e:
        print(f"Error loading detailed results locally: {e}")

elif evaluation_result:
     print("\nEvaluation completed, but no metrics summary was found in the result object.")
     print(f"Check the local output path for details: {OUTPUT_PATH}")
else:
    print("\nEvaluation did not run successfully or produced no result object.")



In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

class HealthcareSyntheticDataGenerator:
    """
    Generate synthetic healthcare data for Azure AI Studio evaluations.
    This class creates realistic synthetic data for various healthcare scenarios.
    """
    
    def __init__(self, seed=42):
        """Initialize the generator with a random seed for reproducibility."""
        self.seed = seed
        np.random.seed(seed)
        
        # Define common constants
        self.current_date = datetime.now()
        
    def generate_patient_readmission_data(self, n_samples=1000, save_path=None):
        """
        Generate synthetic patient data for readmission prediction.
        
        Parameters:
        -----------
        n_samples : int
            Number of patient records to generate
        save_path : str, optional
            Path to save the generated data (CSV)
            
        Returns:
        --------
        DataFrame
            Synthetic patient dataset
        """
        # Create patient IDs
        patient_ids = [f"P{i:07d}" for i in range(n_samples)]
        
        # Demographics
        age = np.random.normal(65, 15, n_samples)
        age = np.clip(age, 18, 100).astype(int)
        gender = np.random.choice(['Male', 'Female'], size=n_samples)
        
        # Vital signs
        systolic_bp = np.random.normal(130, 20, n_samples)
        diastolic_bp = np.random.normal(80, 15, n_samples)
        heart_rate = np.random.normal(75, 15, n_samples)
        respiratory_rate = np.random.normal(16, 4, n_samples)
        temperature = np.random.normal(37, 0.7, n_samples)
        oxygen_saturation = np.random.normal(96, 3, n_samples)
        oxygen_saturation = np.clip(oxygen_saturation, 70, 100)
        
        # Lab values
        glucose = np.random.normal(110, 40, n_samples)
        hemoglobin = np.random.normal(14, 2, n_samples)
        white_blood_cells = np.random.normal(7.5, 3, n_samples)
        platelet_count = np.random.normal(250, 100, n_samples)
        sodium = np.random.normal(140, 5, n_samples)
        potassium = np.random.normal(4.2, 0.5, n_samples)
        creatinine = np.random.normal(1.0, 0.5, n_samples)
        creatinine = np.clip(creatinine, 0.5, 5.0)
        
        # Medical history (binary features)
        hypertension = np.random.choice([0, 1], size=n_samples, p=[0.6, 0.4])
        diabetes = np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25])
        coronary_artery_disease = np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
        heart_failure = np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15])
        stroke_history = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])
        copd = np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15])
        
        # Generate risk factors that will influence outcome
        # Fixed version: using element-wise operators (&, |) instead of logical operators (and, or)
        risk_score = (
            0.7 * (age > 75) +
            0.6 * (systolic_bp > 160) +
            0.5 * (diastolic_bp > 100) +
            0.5 * (heart_rate > 100) +
            0.8 * (respiratory_rate > 24) +
            0.7 * (temperature > 38.5) +
            0.9 * (oxygen_saturation < 90) +
            0.4 * (glucose > 200) +
            0.4 * (hemoglobin < 10) +
            0.4 * (white_blood_cells > 12) +
            0.3 * ((sodium < 135) | (sodium > 145)) +  # Fixed line with | instead of or
            0.4 * ((potassium < 3.5) | (potassium > 5.5)) +  # Fixed line with | instead of or
            0.6 * (creatinine > 1.5) +
            0.3 * hypertension +
            0.4 * diabetes +
            0.5 * coronary_artery_disease +
            0.6 * heart_failure +
            0.5 * stroke_history +
            0.4 * copd
        )
        
        # Add some random noise to risk score
        risk_score += np.random.normal(0, 0.5, n_samples)
        
        # Generate target variable: readmission within 30 days
        readmission_probability = 1 / (1 + np.exp(-(-4 + risk_score * 1.5)))
        readmission = np.random.binomial(1, readmission_probability)
        
        # Length of stay in hospital
        length_of_stay = np.random.lognormal(1.5, 0.6, n_samples).astype(int)
        length_of_stay = np.clip(length_of_stay, 1, 30)
        
        # Admission dates
        admission_dates = []
        for i in range(n_samples):
            days_ago = np.random.randint(40, 400)
            admission_date = (self.current_date - timedelta(days=days_ago)).strftime('%Y-%m-%d')
            admission_dates.append(admission_date)
        
        # Create dataframe
        data = pd.DataFrame({
            'patient_id': patient_ids,
            'age': age,
            'gender': gender,
            'systolic_bp': systolic_bp,
            'diastolic_bp': diastolic_bp,
            'heart_rate': heart_rate,
            'respiratory_rate': respiratory_rate,
            'temperature': temperature,
            'oxygen_saturation': oxygen_saturation,
            'glucose': glucose,
            'hemoglobin': hemoglobin,
            'white_blood_cells': white_blood_cells,
            'platelet_count': platelet_count,
            'sodium': sodium,
            'potassium': potassium,
            'creatinine': creatinine,
            'hypertension': hypertension,
            'diabetes': diabetes,
            'coronary_artery_disease': coronary_artery_disease,
            'heart_failure': heart_failure,
            'stroke_history': stroke_history,
            'copd': copd,
            'length_of_stay': length_of_stay,
            'admission_date': admission_dates,
            'readmission': readmission
        })
        
        # Save data if path is provided
        if save_path:
            data.to_csv(save_path, index=False)
            print(f"Saved readmission data to {save_path}")
            
            # Also save as JSON Lines for Azure AI Studio
            jsonl_path = save_path.replace('.csv', '.jsonl')
            with open(jsonl_path, 'w') as f:
                for _, row in data.iterrows():
                    # Convert row to dict and then to JSON
                    f.write(json.dumps(row.to_dict()) + '\n')
            print(f"Saved readmission data as JSONL to {jsonl_path}")
        
        return data
    
    def generate_medical_imaging_data(self, n_samples=500, save_path=None):
        """
        Generate synthetic metadata for medical imaging analysis.
        
        Parameters:
        -----------
        n_samples : int
            Number of imaging records to generate
        save_path : str, optional
            Path to save the generated data (CSV)
            
        Returns:
        --------
        DataFrame
            Synthetic imaging metadata
        """
        # Create image IDs
        image_ids = [f"IMG{i:07d}" for i in range(n_samples)]
        
        # Patient demographics
        patient_ids = [f"P{np.random.randint(0, 10000):07d}" for _ in range(n_samples)]
        age = np.random.normal(60, 15, n_samples)
        age = np.clip(age, 18, 100).astype(int)
        gender = np.random.choice(['Male', 'Female'], size=n_samples)
        
        # Imaging characteristics
        modality = np.random.choice(['X-ray', 'CT', 'MRI', 'Ultrasound'], size=n_samples, 
                                   p=[0.4, 0.3, 0.2, 0.1])
        
        body_part = []
        for mod in modality:
            if mod == 'X-ray':
                body_part.append(np.random.choice(['Chest', 'Abdomen', 'Extremity'], p=[0.7, 0.2, 0.1]))
            elif mod == 'CT':
                body_part.append(np.random.choice(['Head', 'Chest', 'Abdomen', 'Pelvis'], p=[0.3, 0.3, 0.3, 0.1]))
            elif mod == 'MRI':
                body_part.append(np.random.choice(['Brain', 'Spine', 'Knee', 'Shoulder'], p=[0.4, 0.3, 0.2, 0.1]))
            else:  # Ultrasound
                body_part.append(np.random.choice(['Abdomen', 'Pelvis', 'Vascular', 'Breast'], p=[0.4, 0.3, 0.2, 0.1]))
        
        # Generate findings based on modality and body part
        findings = []
        abnormal = []
        urgency = []
        
        for i in range(n_samples):
            mod = modality[i]
            part = body_part[i]
            
            # Determine if there's an abnormality
            is_abnormal = np.random.choice([0, 1], p=[0.6, 0.4])
            abnormal.append(is_abnormal)
            
            if is_abnormal:
                if mod == 'X-ray' and part == 'Chest':
                    finding = np.random.choice([
                        'Pulmonary nodule', 'Pneumonia', 'Pleural effusion', 
                        'Atelectasis', 'Cardiomegaly'
                    ])
                elif mod == 'CT' and part == 'Head':
                    finding = np.random.choice([
                        'Intracranial hemorrhage', 'Mass lesion', 'Infarct', 
                        'Subdural hematoma', 'Hydrocephalus'
                    ])
                elif mod == 'MRI' and part == 'Brain':
                    finding = np.random.choice([
                        'Multiple sclerosis plaques', 'Tumor', 'Stroke', 
                        'Microhemorrhages', 'Atrophy'
                    ])
                else:
                    finding = np.random.choice([
                        'Mass', 'Inflammation', 'Fluid collection', 
                        'Fracture', 'Degenerative changes'
                    ])
                
                # Determine urgency based on finding
                if finding in ['Intracranial hemorrhage', 'Infarct', 'Stroke', 'Pneumonia']:
                    urg = np.random.choice(['High', 'Critical'], p=[0.7, 0.3])
                else:
                    urg = np.random.choice(['Low', 'Medium', 'High'], p=[0.5, 0.4, 0.1])
            else:
                finding = 'No significant findings'
                urg = 'None'
            
            findings.append(finding)
            urgency.append(urg)
        
        # Scan dates
        scan_dates = []
        for i in range(n_samples):
            days_ago = np.random.randint(1, 400)
            scan_date = (self.current_date - timedelta(days=days_ago)).strftime('%Y-%m-%d')
            scan_dates.append(scan_date)
        
        # Create dataframe
        data = pd.DataFrame({
            'image_id': image_ids,
            'patient_id': patient_ids,
            'age': age,
            'gender': gender,
            'modality': modality,
            'body_part': body_part,
            'scan_date': scan_dates,
            'abnormal': abnormal,
            'finding': findings,
            'urgency': urgency
        })
        
        # Save data if path is provided
        if save_path:
            data.to_csv(save_path, index=False)
            print(f"Saved imaging data to {save_path}")
            
            # Also save as JSON Lines for Azure AI Studio
            jsonl_path = save_path.replace('.csv', '.jsonl')
            with open(jsonl_path, 'w') as f:
                for _, row in data.iterrows():
                    # Convert row to dict and then to JSON
                    f.write(json.dumps(row.to_dict()) + '\n')
            print(f"Saved imaging data as JSONL to {jsonl_path}")
        
        return data
    
    def generate_clinical_notes_data(self, n_samples=300, save_path=None):
        """
        Generate synthetic clinical notes data with entities for NLP evaluation.
        
        Parameters:
        -----------
        n_samples : int
            Number of clinical notes to generate
        save_path : str, optional
            Path to save the generated data (CSV)
            
        Returns:
        --------
        DataFrame
            Synthetic clinical notes with entity annotations
        """
        # Create note IDs
        note_ids = [f"NOTE{i:07d}" for i in range(n_samples)]
        
        # Patient demographics
        patient_ids = [f"P{np.random.randint(0, 10000):07d}" for _ in range(n_samples)]
        age = np.random.normal(60, 15, n_samples)
        age = np.clip(age, 18, 100).astype(int)
        gender = np.random.choice(['Male', 'Female'], size=n_samples)
        
        # Note types
        note_types = np.random.choice([
            'Progress Note', 'Discharge Summary', 'Radiology Report', 
            'Consult Note', 'Emergency Department Note'
        ], size=n_samples)
        
        # Define templates for clinical notes
        progress_note_template = """
SUBJECTIVE: Patient is a {age}-year-old {gender} with a history of {conditions}. Patient {complaint_status} complaining of {complaints}. {additional_symptoms}

OBJECTIVE: 
Vital Signs: Temperature {temp} C, Heart Rate {hr} bpm, Respiratory Rate {rr} breaths/min, Blood Pressure {bp} mmHg, SpO2 {o2}% on {o2_device}.
Physical Exam: {exam_findings}

ASSESSMENT: {assessment}

PLAN:
1. {plan1}
2. {plan2}
3. {plan3}
"""
        
        discharge_template = """
DISCHARGE SUMMARY
Date of Admission: {admit_date}
Date of Discharge: {discharge_date}
Length of Stay: {los} days

ADMISSION DIAGNOSIS: {admission_dx}
DISCHARGE DIAGNOSIS: {discharge_dx}

BRIEF HOSPITAL COURSE:
Patient is a {age}-year-old {gender} admitted for {admission_reason}. During hospitalization, patient received {treatments}. {complications}

DISCHARGE MEDICATIONS:
1. {med1} {dose1} {freq1}
2. {med2} {dose2} {freq2}
3. {med3} {dose3} {freq3}

FOLLOW-UP:
1. Follow up with {provider1} in {followup1} weeks.
2. Follow up with {provider2} in {followup2} weeks if symptoms persist.
"""
        
        radiology_template = """
EXAM: {exam_type}
CLINICAL INDICATION: {indication}

TECHNIQUE: {technique}

FINDINGS:
{findings}

IMPRESSION:
{impression}
"""
        
        # Generate clinical notes and extract entities
        notes = []
        entities = []
        
        # Define possible values for templates
        conditions_list = ['hypertension', 'diabetes mellitus type 2', 'hyperlipidemia', 'coronary artery disease', 
                         'congestive heart failure', 'COPD', 'asthma', 'atrial fibrillation', 'CKD stage 3', 'osteoarthritis']
        complaint_list = ['chest pain', 'shortness of breath', 'abdominal pain', 'headache', 'dizziness', 
                        'nausea and vomiting', 'back pain', 'joint pain', 'fever', 'cough']
        symptom_list = ['fatigue', 'weight loss', 'poor appetite', 'night sweats', 'palpitations', 
                      'edema', 'rash', 'blurred vision', 'numbness', 'weakness']
        
        medications = ['lisinopril', 'metoprolol', 'atorvastatin', 'metformin', 'aspirin', 
                     'furosemide', 'amlodipine', 'levothyroxine', 'albuterol', 'sertraline']
        doses = ['10 mg', '25 mg', '20 mg', '500 mg', '81 mg', '40 mg', '5 mg', '100 mcg', '90 mcg', '50 mg']
        frequencies = ['daily', 'twice daily', 'three times daily', 'four times daily', 'every 12 hours', 
                      'every 8 hours', 'every 6 hours', 'weekly', 'as needed', 'at bedtime']
        
        for i in range(n_samples):
            note_type = note_types[i]
            patient_age = age[i]
            patient_gender = gender[i]
            
            # Generate clinical note based on note type
            if note_type == 'Progress Note':
                # Generate components for progress note
                patient_conditions = ', '.join(np.random.choice(conditions_list, size=np.random.randint(1, 4), replace=False))
                complaint_status = np.random.choice(['is', 'is not'])
                patient_complaints = ', '.join(np.random.choice(complaint_list, size=np.random.randint(1, 3), replace=False))
                add_symptoms = 'Patient also reports ' + ', '.join(np.random.choice(symptom_list, size=np.random.randint(0, 3), replace=False)) if np.random.random() > 0.3 else ''
                
                temp = round(np.random.normal(37, 0.8), 1)
                hr = int(np.random.normal(80, 15))
                rr = int(np.random.normal(16, 4))
                sbp = int(np.random.normal(130, 20))
                dbp = int(np.random.normal(80, 10))
                bp = f"{sbp}/{dbp}"
                o2 = int(np.random.normal(97, 2))
                o2_device = np.random.choice(['room air', 'nasal cannula at 2L', 'nasal cannula at 4L', 'face mask at 10L'])
                
                exam_findings = np.random.choice([
                    'Patient appears well. Heart with regular rate and rhythm, no murmurs. Lungs clear to auscultation bilaterally.',
                    'Patient in mild distress. Heart rate regular, S1/S2 normal. Lungs with faint crackles at bases.',
                    'Patient comfortable. Cardiac exam with regular rhythm. Lungs with scattered wheezes.',
                    'Patient alert and oriented. Cardiovascular exam unremarkable. Respiratory exam shows decreased breath sounds.'
                ])
                
                assessment = np.random.choice([
                    f'1. {np.random.choice(conditions_list)} - stable\n2. {np.random.choice(complaint_list)} - improved',
                    f'1. {np.random.choice(conditions_list)} - worsening\n2. {np.random.choice(complaint_list)} - unresolved',
                    f'1. {np.random.choice(conditions_list)} - controlled\n2. {np.random.choice(complaint_list)} - resolved',
                    f'1. Acute {np.random.choice(complaint_list)}\n2. Chronic {np.random.choice(conditions_list)}'
                ])
                
                plan1 = np.random.choice([
                    f'Continue {np.random.choice(medications)} {np.random.choice(doses)} {np.random.choice(frequencies)}',
                    f'Increase {np.random.choice(medications)} to {np.random.choice(doses)} {np.random.choice(frequencies)}',
                    f'Start {np.random.choice(medications)} {np.random.choice(doses)} {np.random.choice(frequencies)}',
                    f'Discontinue {np.random.choice(medications)}'
                ])
                
                plan2 = np.random.choice([
                    f'Order {np.random.choice(["CBC", "CMP", "Chest X-ray", "ECG", "CT scan", "MRI"])}',
                    f'Refer to {np.random.choice(["Cardiology", "Pulmonology", "Neurology", "Gastroenterology", "Nephrology"])}',
                    f'Schedule follow-up in {np.random.choice(["1 week", "2 weeks", "1 month", "3 months"])}',
                    f'Educate patient on {np.random.choice(["diet", "exercise", "medication adherence", "smoking cessation"])}'
                ])
                
                plan3 = np.random.choice([
                    f'Monitor {np.random.choice(["blood pressure", "glucose levels", "symptoms", "weight"])} at home',
                    f'Return to clinic if {np.random.choice(["symptoms worsen", "fever develops", "no improvement in 48 hours", "any concerns"])}',
                    f'Continue current {np.random.choice(["diet", "exercise regimen", "home medications", "treatment plan"])}',
                    f'Discussed {np.random.choice(["risk factors", "prognosis", "treatment options", "side effects"])} with patient'
                ])
                
                # Format the note
                note = progress_note_template.format(
                    age=patient_age, gender=patient_gender, conditions=patient_conditions,
                    complaint_status=complaint_status, complaints=patient_complaints,
                    additional_symptoms=add_symptoms, temp=temp, hr=hr, rr=rr, bp=bp,
                    o2=o2, o2_device=o2_device, exam_findings=exam_findings,
                    assessment=assessment, plan1=plan1, plan2=plan2, plan3=plan3
                )
                
                # Extract entities
                note_entities = []
                
                # Demographic entities
                note_entities.append({"type": "DEMOGRAPHIC", "text": f"{patient_age}-year-old {patient_gender}", "start": note.find(f"{patient_age}-year-old {patient_gender}")})
                
                # Condition entities
                for condition in patient_conditions.split(', '):
                    pos = note.find(condition)
                    if pos >= 0:
                        note_entities.append({"type": "CONDITION", "text": condition, "start": pos})
                
                # Symptom entities
                for symptom in patient_complaints.split(', '):
                    pos = note.find(symptom)
                    if pos >= 0:
                        note_entities.append({"type": "SYMPTOM", "text": symptom, "start": pos})
                
                # Medication entities
                for med in medications:
                    pos = note.find(med)
                    if pos >= 0:
                        # Try to find the dose and frequency
                        end_pos = note.find('\n', pos)
                        if end_pos < 0:
                            end_pos = len(note)
                        med_text = note[pos:end_pos]
                        note_entities.append({"type": "MEDICATION", "text": med_text, "start": pos})
                        
            else:
                # Default to a shorter generic note for other note types
                note = f"Patient: {patient_age}-year-old {patient_gender}\nNote Type: {note_type}\n\n"
                note += np.random.choice([
                    f"Patient seen for {np.random.choice(complaint_list)}. History of {np.random.choice(conditions_list)}. " +
                    f"Prescribed {np.random.choice(medications)} {np.random.choice(doses)} {np.random.choice(frequencies)}.",
                    
                    f"Follow-up visit for {np.random.choice(conditions_list)}. Patient reports {np.random.choice(symptom_list)}. " +
                    f"Continue {np.random.choice(medications)} {np.random.choice(doses)} {np.random.choice(frequencies)}.",
                    
                    f"Emergency visit for acute {np.random.choice(complaint_list)}. Diagnostic tests performed. " +
                    f"Treatment with {np.random.choice(medications)} initiated."
                ])
                
                # Extract entities
                note_entities = []
                
                # Demographic entities
                note_entities.append({"type": "DEMOGRAPHIC", "text": f"{patient_age}-year-old {patient_gender}", "start": note.find(f"{patient_age}-year-old {patient_gender}")})
                
                # Simple entity extraction based on known medications and conditions
                for condition in conditions_list:
                    pos = note.find(condition)
                    if pos >= 0:
                        note_entities.append({"type": "CONDITION", "text": condition, "start": pos})
                
                for med in medications:
                    pos = note.find(med)
                    if pos >= 0:
                        note_entities.append({"type": "MEDICATION", "text": med, "start": pos})
            
            # Add the note and entities to our lists
            notes.append(note)
            entities.append(note_entities)
        
        # Create note dates
        note_dates = []
        for i in range(n_samples):
            days_ago = np.random.randint(1, 400)
            note_date = (self.current_date - timedelta(days=days_ago)).strftime('%Y-%m-%d')
            note_dates.append(note_date)
        
        # Create dataframe
        data = pd.DataFrame({
            'note_id': note_ids,
            'patient_id': patient_ids,
            'age': age,
            'gender': gender,
            'note_type': note_types,
            'note_date': note_dates,
            'note_text': notes,
            'entities': entities
        })
        
        # Save data if path is provided
        if save_path:
            # For the entities, we need to convert to JSON strings
            data_to_save = data.copy()
            data_to_save['entities'] = data_to_save['entities'].apply(json.dumps)
            data_to_save.to_csv(save_path, index=False)
            print(f"Saved clinical notes data to {save_path}")
            
            # Also save as JSON Lines for Azure AI Studio
            jsonl_path = save_path.replace('.csv', '.jsonl')
            with open(jsonl_path, 'w') as f:
                for _, row in data_to_save.iterrows():
                    # Convert row to dict and then to JSON
                    f.write(json.dumps(row.to_dict()) + '\n')
            print(f"Saved clinical notes data as JSONL to {jsonl_path}")
        
        return data
    
    def generate_evaluation_dataset_for_azure_studio(self, output_dir="azure_eval_data"):
        """
        Generate a comprehensive evaluation dataset for Azure AI Studio healthcare models.
        
        Parameters:
        -----------
        output_dir : str
            Directory to save the generated data
            
        Returns:
        --------
        dict
            Paths to generated datasets
        """
        os.makedirs(output_dir, exist_ok=True)
        
        # Generate all types of data
        print("Generating patient readmission data...")
        readmission_path = os.path.join(output_dir, "readmission_data.csv")
        readmission_data = self.generate_patient_readmission_data(n_samples=1000, save_path=readmission_path)
        
        print("Generating medical imaging data...")
        imaging_path = os.path.join(output_dir, "medical_imaging_data.csv")
        imaging_data = self.generate_medical_imaging_data(n_samples=500, save_path=imaging_path)
        
        print("Generating clinical notes data...")
        notes_path = os.path.join(output_dir, "clinical_notes_data.csv")
        notes_data = self.generate_clinical_notes_data(n_samples=300, save_path=notes_path)
        
        # Create data visualizations to help with Azure AI Studio evaluation setup
        self._create_data_visualizations(readmission_data, imaging_data, notes_data, output_dir)
        
        return {
            "readmission_data": readmission_path,
            "imaging_data": imaging_path,
            "clinical_notes_data": notes_path
        }
    
    def _create_data_visualizations(self, readmission_data, imaging_data, notes_data, output_dir):
        """Create visualizations of the synthetic data for documentation."""
        # Create visualizations directory
        viz_dir = os.path.join(output_dir, "visualizations")
        os.makedirs(viz_dir, exist_ok=True)
        
        # 1. Readmission data visualizations
        plt.figure(figsize=(10, 6))
        sns.countplot(x='readmission', data=readmission_data)
        plt.title('Distribution of Readmission Outcomes')
        plt.xlabel('Readmission (0=No, 1=Yes)')
        plt.ylabel('Count')
        plt.savefig(os.path.join(viz_dir, 'readmission_distribution.png'))
        
        plt.figure(figsize=(12, 8))
        medical_history = ['hypertension', 'diabetes', 'coronary_artery_disease', 
                          'heart_failure', 'stroke_history', 'copd']
        
        readmission_rates = []
        conditions = []
        
        for condition in medical_history:
            rate = readmission_data[readmission_data[condition] == 1]['readmission'].mean() * 100
            readmission_rates.append(rate)
            conditions.append(condition)
        
        condition_df = pd.DataFrame({'condition': conditions, 'readmission_rate': readmission_rates})
        sns.barplot(data=condition_df, x='condition', y='readmission_rate')
        plt.title('Readmission Rate by Medical Condition')
        plt.ylabel('Readmission Rate (%)')
        plt.xlabel('Medical Condition')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'readmission_by_condition.png'))
        
        # 2. Imaging data visualizations
        plt.figure(figsize=(10, 6))
        sns.countplot(x='modality', data=imaging_data)
        plt.title('Distribution of Imaging Modalities')
        plt.xlabel('Modality')
        plt.ylabel('Count')
        plt.savefig(os.path.join(viz_dir, 'imaging_modalities.png'))
        
        plt.figure(figsize=(10, 6))
        sns.countplot(x='abnormal', data=imaging_data)
        plt.title('Distribution of Abnormal Findings')
        plt.xlabel('Abnormal Finding (0=No, 1=Yes)')
        plt.ylabel('Count')
        plt.savefig(os.path.join(viz_dir, 'abnormal_findings.png'))
        
        # 3. Clinical notes visualizations
        plt.figure(figsize=(10, 6))
        sns.countplot(x='note_type', data=notes_data)
        plt.title('Distribution of Note Types')
        plt.xlabel('Note Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'note_types.png'))
        
        # Entity type distribution
        entity_types = []
        for entities in notes_data['entities']:
            for entity in entities:
                entity_types.append(entity['type'])
        
        entity_df = pd.DataFrame({'entity_type': entity_types})
        
        plt.figure(figsize=(10, 6))
        sns.countplot(x='entity_type', data=entity_df)
        plt.title('Distribution of Entity Types')
        plt.xlabel('Entity Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(viz_dir, 'entity_types.png'))
        
        print(f"Data visualizations saved to {viz_dir}")

# Example usage
if __name__ == "__main__":
    generator = HealthcareSyntheticDataGenerator(seed=42)
    
    # Generate comprehensive evaluation dataset
    dataset_paths = generator.generate_evaluation_dataset_for_azure_studio("azure_healthcare_eval_data")
    
    print("\nGenerated datasets:")
    for dataset_name, path in dataset_paths.items():
        print(f"- {dataset_name}: {path}")
    
    print("\nData is ready for use in Azure AI Studio evaluations!")

In [ ]:
"""
Healthcare Model Scoring Script for Azure Machine Learning
This script is used for model inferencing in Azure ML deployments.
"""

import os
import json
import numpy as np
import pandas as pd
import joblib
import time
from datetime import datetime
import logging
from inference_schema.schema_decorators import input_schema, output_schema
from inference_schema.parameter_types.numpy_parameter_type import NumpyParameterType
from inference_schema.parameter_types.standard_py_parameter_type import StandardPythonParameterType

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("healthcare_model_inference")

# Define input and output schemas
class HealthcareModelParams:
    """
    Input parameters for the healthcare model.
    These schemas will be used by Azure ML to generate OpenAPI specifications.
    """
    # Input schema for patient readmission model
    readmission_schema = {
        'age': NumpyParameterType(np.array([0.0], dtype=np.float64)),
        'gender': StandardPythonParameterType("Male"),
        'systolic_bp': NumpyParameterType(np.array([120.0], dtype=np.float64)),
        'diastolic_bp': NumpyParameterType(np.array([80.0], dtype=np.float64)),
        'heart_rate': NumpyParameterType(np.array([70.0], dtype=np.float64)),
        'respiratory_rate': NumpyParameterType(np.array([16.0], dtype=np.float64)),
        'temperature': NumpyParameterType(np.array([98.6], dtype=np.float64)),
        'oxygen_saturation': NumpyParameterType(np.array([98.0], dtype=np.float64)),
        'glucose': NumpyParameterType(np.array([90.0], dtype=np.float64)),
        'hemoglobin': NumpyParameterType(np.array([13.5], dtype=np.float64)),
        'white_blood_cells': NumpyParameterType(np.array([7.0], dtype=np.float64)),
        'platelet_count': NumpyParameterType(np.array([250.0], dtype=np.float64)),
        'sodium': NumpyParameterType(np.array([140.0], dtype=np.float64)),
        'potassium': NumpyParameterType(np.array([4.0], dtype=np.float64)),
        'creatinine': NumpyParameterType(np.array([1.0], dtype=np.float64)),
        'hypertension': NumpyParameterType(np.array([1], dtype=np.int32)),
        'diabetes': NumpyParameterType(np.array([0], dtype=np.int32)),
        'coronary_artery_disease': NumpyParameterType(np.array([0], dtype=np.int32)),
        'heart_failure': NumpyParameterType(np.array([0], dtype=np.int32)),
        'stroke_history': NumpyParameterType(np.array([0], dtype=np.int32)),
        'copd': NumpyParameterType(np.array([0], dtype=np.int32)),
        'length_of_stay': NumpyParameterType(np.array([5], dtype=np.int32))
    }
    
    # Input schema for medical imaging model
    imaging_schema = {
        'age': NumpyParameterType(float),
        'gender': StandardPythonParameterType(str),
        'modality': StandardPythonParameterType(str),
        'body_part': StandardPythonParameterType(str)
    }
    
    # Input schema for clinical NLP model
    nlp_schema = {
        'note_text': StandardPythonParameterType(str),
        'note_type': StandardPythonParameterType(str)
    }

# Define the global variables for model and preprocessing objects
model = None
model_type = None
feature_names = None
categorical_columns = ['gender']
numerical_columns = None  # Will be set during init()
preprocessing = None
label_encoders = {}
model_metadata = {}

def init():
    """
    Initialize the model when the container starts.
    This function is called when the container is initialized.
    """
    global model, model_type, feature_names, numerical_columns, preprocessing, label_encoders, model_metadata
    
    # Log model initialization start
    logger.info("Model initialization started")
    start_time = time.time()
    
    try:
        # Get the path to the model directory
        model_dir = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "models")
        
        # Load model metadata
        metadata_path = os.path.join(model_dir, "model_metadata.json")
        if os.path.exists(metadata_path):
            with open(metadata_path, "r") as f:
                model_metadata = json.load(f)
                model_type = model_metadata.get("model_type", "readmission")
                logger.info(f"Loaded model metadata. Model type: {model_type}")
        else:
            # Default to readmission model if metadata not found
            model_type = "readmission"
            logger.warning("Model metadata not found. Defaulting to readmission model.")
        
        # Load the appropriate model
        model_path = os.path.join(model_dir, "model.pkl")
        logger.info(f"Loading model from {model_path}")
        model = joblib.load(model_path)
        
        # Load feature names, if available
        feature_names_path = os.path.join(model_dir, "feature_names.csv")
        if os.path.exists(feature_names_path):
            feature_names = pd.read_csv(feature_names_path).iloc[:, 0].tolist()
            logger.info(f"Loaded {len(feature_names)} feature names")
        
        # Load preprocessing objects if available
        preprocessing_path = os.path.join(model_dir, "preprocessing.pkl")
        if os.path.exists(preprocessing_path):
            preprocessing = joblib.load(preprocessing_path)
            logger.info("Loaded preprocessing pipeline")
        
        # Load label encoders for categorical features
        encoders_path = os.path.join(model_dir, "label_encoders.pkl")
        if os.path.exists(encoders_path):
            label_encoders = joblib.load(encoders_path)
            logger.info(f"Loaded {len(label_encoders)} label encoders for categorical features")
        
        # Set numerical columns based on feature names and categorical columns
        if feature_names:
            numerical_columns = [col for col in feature_names if col not in categorical_columns]
            logger.info(f"Identified {len(numerical_columns)} numerical columns")
        
        # Log successful initialization
        initialization_time = time.time() - start_time
        logger.info(f"Model initialization completed successfully in {initialization_time:.2f} seconds")
    
    except Exception as e:
        logger.error(f"Error during model initialization: {str(e)}")
        raise

def preprocess_input(data_dict):
    """
    Preprocess the input data for model prediction.
    
    Parameters:
    -----------
    data_dict : dict
        Dictionary containing input data
    
    Returns:
    --------
    pd.DataFrame
        Preprocessed data ready for model prediction
    """
    try:
        # Convert input dictionary to DataFrame
        if isinstance(data_dict, list):
            # Handle batch prediction (list of dictionaries)
            input_df = pd.DataFrame(data_dict)
        else:
            # Handle single prediction (single dictionary)
            input_df = pd.DataFrame([data_dict])
        
        logger.info(f"Input data shape: {input_df.shape}")
        
        # Handle different model types
        if model_type == "readmission":
            # Convert gender to categorical if it's a string
            if 'gender' in input_df.columns and input_df['gender'].dtype == 'object':
                input_df['gender'] = input_df['gender'].map({'Male': 1, 'Female': 0})
                logger.info("Converted gender to binary feature")
            
            # Ensure all expected features are present
            if feature_names:
                for feature in feature_names:
                    if feature not in input_df.columns:
                        # Handle missing features
                        logger.warning(f"Feature {feature} missing from input, adding with default value 0")
                        input_df[feature] = 0
                
                # Select only the features expected by the model in the right order
                input_df = input_df[feature_names]
        
        elif model_type == "imaging":
            # Process imaging data
            if 'gender' in input_df.columns and input_df['gender'].dtype == 'object':
                input_df['gender'] = input_df['gender'].map({'Male': 1, 'Female': 0})
            
            # One-hot encode modality and body_part if needed
            if 'modality' in input_df.columns:
                modality_dummies = pd.get_dummies(input_df['modality'], prefix='modality')
                input_df = pd.concat([input_df, modality_dummies], axis=1)
                input_df.drop('modality', axis=1, inplace=True)
            
            if 'body_part' in input_df.columns:
                body_part_dummies = pd.get_dummies(input_df['body_part'], prefix='body_part')
                input_df = pd.concat([input_df, body_part_dummies], axis=1)
                input_df.drop('body_part', axis=1, inplace=True)
        
        elif model_type == "nlp":
            # For NLP models, ensure text field is present
            if 'note_text' not in input_df.columns:
                logger.error("Required field 'note_text' missing from input")
                raise ValueError("Required field 'note_text' missing from input")
            
            # If preprocessing is available, apply it
            if preprocessing:
                # This could be a tokenizer, vectorizer, etc.
                return preprocessing.transform(input_df['note_text'].values)
        
        # Apply preprocessing if available and not an NLP model
        if preprocessing and model_type != "nlp":
            logger.info("Applying preprocessing pipeline")
            input_data = preprocessing.transform(input_df)
        else:
            # No preprocessing or already applied for NLP
            input_data = input_df
        
        return input_data
    
    except Exception as e:
        logger.error(f"Error during input preprocessing: {str(e)}")
        raise

def postprocess_output(prediction, input_data):
    """
    Postprocess model outputs to create the final prediction response.
    
    Parameters:
    -----------
    prediction : array-like
        Raw model predictions
    input_data : dict or DataFrame
        Original input data
    
    Returns:
    --------
    dict
        Processed prediction result with additional metadata
    """
    try:
        # Handle different model types
        if model_type == "readmission":
            # Get probabilities if available
            if hasattr(model, 'predict_proba'):
                probabilities = model.predict_proba(prediction)
                # For binary classification, get the probability of positive class
                if probabilities.shape[1] == 2:
                    readmission_probability = probabilities[:, 1].tolist()
                else:
                    readmission_probability = probabilities.tolist()
            else:
                readmission_probability = None
            
            # Prepare the result
            result = {
                "prediction": prediction.tolist() if isinstance(prediction, np.ndarray) else prediction,
                "probability": readmission_probability,
                "timestamp": datetime.now().isoformat(),
                "model_version": model_metadata.get("version", "unknown"),
                "model_type": model_type
            }
            
            # Add risk level based on probability
            if readmission_probability and isinstance(readmission_probability[0], (int, float)):
                risk_levels = []
                for prob in readmission_probability:
                    if isinstance(prob, list):
                        # For multi-class, use the highest probability
                        prob = max(prob)
                    
                    if prob < 0.3:
                        risk_levels.append("Low")
                    elif prob < 0.7:
                        risk_levels.append("Medium")
                    else:
                        risk_levels.append("High")
                
                result["risk_level"] = risk_levels
            
            return result
        
        elif model_type == "imaging":
            # For imaging models, interpret the prediction based on model output type
            if isinstance(prediction, np.ndarray) and prediction.ndim > 1:
                # Handle multi-class outputs
                predicted_class = np.argmax(prediction, axis=1).tolist()
                probabilities = prediction.tolist()
            else:
                # Handle binary outputs
                predicted_class = prediction.tolist() if isinstance(prediction, np.ndarray) else prediction
                probabilities = None
            
            # Map class indices to finding labels if available
            finding_labels = model_metadata.get("classes", [])
            if finding_labels and isinstance(predicted_class, list):
                findings = [finding_labels[cls] if cls < len(finding_labels) else f"Class_{cls}" 
                          for cls in predicted_class]
            else:
                findings = predicted_class
            
            return {
                "predicted_finding": findings,
                "probabilities": probabilities,
                "timestamp": datetime.now().isoformat(),
                "model_version": model_metadata.get("version", "unknown"),
                "model_type": model_type
            }
        
        elif model_type == "nlp":
            # For NLP models, the output could be extracted entities or classifications
            if "entities" in model_metadata.get("output_type", ""):
                # Process extracted entities (e.g., medications, conditions)
                return {
                    "entities": prediction,
                    "timestamp": datetime.now().isoformat(),
                    "model_version": model_metadata.get("version", "unknown"),
                    "model_type": model_type
                }
            else:
                # Default classification output
                return {
                    "classification": prediction.tolist() if isinstance(prediction, np.ndarray) else prediction,
                    "timestamp": datetime.now().isoformat(),
                    "model_version": model_metadata.get("version", "unknown"),
                    "model_type": model_type
                }
        
        else:
            # Generic output for other model types
            return {
                "prediction": prediction.tolist() if isinstance(prediction, np.ndarray) else prediction,
                "timestamp": datetime.now().isoformat(),
                "model_version": model_metadata.get("version", "unknown"),
                "model_type": model_type
            }
    
    except Exception as e:
        logger.error(f"Error during output postprocessing: {str(e)}")
        raise

@input_schema('data', StandardPythonParameterType({'data': list}))
@output_schema(StandardPythonParameterType({'prediction': list, 'probability': list}))
def run(data):
    """
    Execute model prediction on the input data.
    
    Parameters:
    -----------
    data : dict
        Input data in JSON format
    
    Returns:
    --------
    dict
        Prediction result
    """
    try:
        logger.info("Starting prediction request")
        start_time = time.time()
        
        # Get the input data from the request
        if isinstance(data, dict) and 'data' in data:
            input_data = data['data']
        else:
            input_data = data
            
        # Log input data summary
        logger.info(f"Received prediction request with {len(input_data) if isinstance(input_data, list) else 1} records")
        
        # Preprocess the input data
        processed_input = preprocess_input(input_data)
        
        # Make the prediction
        logger.info("Making prediction")
        raw_prediction = model.predict(processed_input)
        
        # Postprocess the prediction result
        result = postprocess_output(raw_prediction, input_data)
        
        # Log completion
        prediction_time = time.time() - start_time
        logger.info(f"Prediction completed successfully in {prediction_time:.2f} seconds")
        
        return result
    
    except Exception as e:
        logger.error(f"Error during prediction: {str(e)}")
        error_message = str(e)
        return {"error": error_message, "timestamp": datetime.now().isoformat()}

# Additional utility functions for special preprocessing cases
def encode_categorical_features(df, feature_encoders=None):
    """
    Encode categorical features using label encoders.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing categorical features
    feature_encoders : dict, optional
        Dictionary of label encoders for each categorical feature
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with encoded categorical features
    dict
        Updated feature encoders
    """
    encoded_df = df.copy()
    encoders = feature_encoders or {}
    
    for column in categorical_columns:
        if column in df.columns:
            if column not in encoders:
                from sklearn.preprocessing import LabelEncoder
                encoders[column] = LabelEncoder()
                encoders[column].fit(df[column].astype(str))
            
            encoded_df[column] = encoders[column].transform(df[column].astype(str))
    
    return encoded_df, encoders

def handle_missing_values(df):
    """
    Handle missing values in the input data.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame potentially containing missing values
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with handled missing values
    """
    # Check for missing values
    if df.isnull().any().any():
        # For numerical columns, impute with median
        for col in df.select_dtypes(include=['number']).columns:
            median_value = df[col].median()
            df[col].fillna(median_value, inplace=True)
        
        # For categorical columns, impute with mode
        for col in df.select_dtypes(include=['object']).columns:
            mode_value = df[col].mode()[0]
            df[col].fillna(mode_value, inplace=True)
    
    return df

In [ ]:
# 1. Imports (ensure these are present)
import os
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,  # Keep if defining environment here, otherwise optional
    CodeConfiguration, # <-- Import CodeConfiguration
)
import json # Keep if needed elsewhere in your script
import logging # Keep if needed elsewhere in your script

# 2. Configure MLClient (replace with your details)
try:
    credential = DefaultAzureCredential()
    #ml_client = MLClient.from_config(credential=credential)
    # Alternatively, explicitly define:
    ml_client = MLClient(
        credential=credential,
        subscription_id = "8855ce15-2cf9-4df6-8f25-33ee12bb4bf3",
        resource_group_name="ai-demos",
        workspace_name="ai-demos-ml",
       #workspace_location="<YOUR_WORKSPACE_LOCATION>" # Optional, if not in config
    )
    print(f"Connected to workspace: {ml_client.workspace_name}")
except Exception as e:
    print(f"Error connecting to Azure ML Workspace: {e}")
    exit()

# 3. Configuration Variables
endpoint_name = "healthevals-endpoint" # Or your desired unique endpoint name
deployment_name = "blue" # Deployment name (e.g., for blue-green)
model_name = "healthevalss" # Ensure this matches your registered model name
# model_version = "4" # Or get the latest version dynamically if needed

instance_type = "Standard_D4_v3" # Example instance type
instance_count = 1

# --- CRITICAL: Choose a suitable Environment ---
# You MUST select an environment that has the necessary packages installed
# (pandas, joblib, scikit-learn, numpy etc. - based on your score.py imports)

# Option A: Use a Curated Environment (Recommended if it fits)
# Find suitable ones via `ml_client.environments.list(registry_name="azureml")` or Studio UI
# Example for scikit-learn 1.1:
#chosen_environment_id = "azureml://registries/azureml/environments/sklearn-1.1/labels/latest"
# Example for a basic Python environment (ensure pandas/joblib etc. are added if needed):
# chosen_environment_id = "azureml://registries/azureml/environments/minimal/labels/latest"

# Option B: Use a Custom Environment registered in your workspace
custom_env_name = "azurehealth-evals"
custom_env_version = "1"
chosen_environment_id = f"{custom_env_name}:{custom_env_version}"
# --- End Environment Choice ---


# 4. Get the Registered Model
try:
    # Get the latest version if version not specified
    registered_model = ml_client.models.get(name=model_name, label="latest") # Or specify version=model_version
    print(f"Using model: {registered_model.name} version {registered_model.version}")
except Exception as e:
    print(f"Error retrieving model '{model_name}': {e}")
    # If the model wasn't registered in the previous step, you might need to register it here first
    # Example:
    model_local_path = '/code/Evals/model_to_register/' # Path where model file(s) are *locally*
    registered_model = ml_client.models.create_or_update(
        Model(name=model_name, path=model_local_path, description="Health Evals Model")
    )
    print(f"Registered model '{registered_model.name}' version {registered_model.version}")
    exit()


# 5. Define the Online Endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Online endpoint for Health Evals model",
    auth_mode="key", # Or "aml_token"
)

print(f"Creating/Updating Endpoint '{endpoint.name}'...")
try:
    endpoint_job = ml_client.online_endpoints.begin_create_or_update(endpoint)
    endpoint_job.wait()
    print("Endpoint created/updated successfully.")
except Exception as e:
    print(f"Endpoint creation/update failed: {e}")
    # Consider if you need to handle existing endpoints or potential conflicts
    exit()


# --- 6. Define the Code Configuration for your score.py ---
code_config = CodeConfiguration(
    code="./src",  # <-- Points to the directory containing score.py
    scoring_script="score.py", # <-- Specifies the entry script file name
)
# --- End Code Configuration Definition ---


# 7. Define the Online Deployment ('blue')
print(f"Defining Deployment '{deployment_name}'...")
deployment = ManagedOnlineDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=registered_model,           # Reference the registered model object
    environment=chosen_environment_id,# *** Use the specific environment ID ***
    code_configuration=code_config,   # *** Include the code configuration ***
    instance_type=instance_type,
    instance_count=instance_count,
)

# 8. Create or Update the Deployment
print(f"Creating/Updating Deployment '{deployment.name}' for Endpoint '{endpoint_name}'...")
try:
    deployment_job = ml_client.online_deployments.begin_create_or_update(deployment)
    # deployment_job.wait() # Use wait() for synchronous execution
    print(f"Deployment job initiated. Status URL: {deployment_job.get_status_url()}")
    # It's often better to poll or check status later for long deployments
    # Or use wait() if you need the script to block until completion:
    deployment_job.wait()
    print(f"Deployment '{deployment.name}' created/updated successfully.")

except Exception as e:
    print(f"Deployment creation/update failed: {e}")
    # Check logs in Azure ML Studio for detailed deployment errors
    exit()


# 9. Allocate Traffic (Example: 100% to blue)
print(f"Setting traffic for endpoint '{endpoint_name}'...")
endpoint = ml_client.online_endpoints.get(name=endpoint_name) # Refresh endpoint object
endpoint.traffic = {deployment_name: 100}
try:
    traffic_job = ml_client.online_endpoints.begin_create_or_update(endpoint)
    traffic_job.wait()
    print(f"Traffic set: {deployment_name}=100%")
except Exception as e:
    print(f"Failed to set traffic: {e}")
    exit()


# 10. Get Endpoint Details for Interaction
print("\n--- Endpoint Details ---")
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
print(f"Endpoint '{endpoint.name}' provisioning state: {endpoint.provisioning_state}")
if endpoint.scoring_uri:
    print(f"Scoring URI: {endpoint.scoring_uri}")
    try:
        endpoint_keys = ml_client.online_endpoints.get_keys(name=endpoint_name)
        primary_key = endpoint_keys.primary_key
        print(f"Primary Key: [REDACTED - Use Key Vault in production]") # Avoid printing keys directly
        print(f"Primary Key: {primary_key}") # Uncomment for testing ONLY
    except Exception as e:
        print(f"Could not retrieve endpoint keys: {e}")
else:
    print("Scoring URI not available yet. Endpoint might still be provisioning.")

print("\nDeployment finished.")

Connected to workspace: ai-demos-ml
Using model: healthevalss version 1
Creating/Updating Endpoint 'healthevals-endpoint'...


Check: endpoint healthevals-endpoint exists


Endpoint created/updated successfully.
Defining Deployment 'blue'...
Creating/Updating Deployment 'blue' for Endpoint 'healthevals-endpoint'...
Deployment creation/update failed: 'LROPoller' object has no attribute 'get_status_url'
Setting traffic for endpoint 'healthevals-endpoint'...
.

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Failed to set traffic: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"The deployments [blue] on which you are trying to set the traffic are in a failed state and don't have ingress. Please fix the deployments and try again.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-3fc526696cce7fdc3f0c600b6ed8fab0-fb42f4a8435d3795-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"DeploymentWeights\":[\"The deployments [blue] on which you are trying to set the traffic are in a failed state and don't have ingress. Please fix the deployments and try again.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurre

: 